# 31. RG3 가스 + 미성형 함께 잡기 — 하연님 가스 모델에 미성형 변수 추가

**하연님 22번 틀 그대로**: 같은 데이터(RG3 591샷: 정상 566 · 가스 17 · 미성형 8), 같은 랜덤포레스트, 같은 검증(관측 구간 빼기 / 과거 구간만 학습), 같은 임계값 규칙(학습 정상의 안쪽 교차검증 99% 분위수)

| 모델 | 변수 | 학습 대상 |
|---|---|---|
| **A_gas_std4** | 하연님 흔들림 4개 (배럴2·호퍼·쿠션·배럴5 직전 10샷 표준편차) | 가스 vs 정상 — **하연님 std4 재현** |
| **B_short_short6** | 미성형 6개 (계량시간 편차·변화, 배압, 계량위치, 쿠션, 충전시간 편차) | 미성형 vs 정상 |
| **C_union_max** | max(A, B) | 둘 중 하나라도 의심 |
| **D_all_in_one** | A + B 변수 10개 | 불량 전체 vs 정상 |

미성형 변수 근거: 11번 RG3 미성형 = 계량시간↑(δ 0.40), 12번 계량 지연 가설 지지(4/5 Run) → labeled 분석에서 나온 가설이라 완전히 독립적인 선택은 아님

**먼저 확인**: A의 'AP 가스'(관측 구간 빼기)가 하연님 결과 **0.1387**과 같으면 재현 성공

In [1]:
import os
import pandas as pd
from rg3_two_type import load, run, metrics
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
OUT = './outputs/rg3_two_type'
os.makedirs(OUT, exist_ok=True)

D = load('../../KAMP-IMM_hayeon/reports/tables')      # 하연님 레포 표 경로 (다르면 수정)

입력: run_shot_feature_candidates.csv + run_label_observation_blocks.csv | RG3 591샷 — {'정상': 566, '가스': 17, '미성형': 8} | 구간: {(3.0, '정상'): 41.0, (3.0, '가스'): nan, (3.0, '미성형'): nan, (4.0, '정상'): 98.0, (4.0, '가스'): 4.0, (4.0, '미성형'): 1.0, (5.0, '정상'): 187.0, (5.0, '가스'): 6.0, (5.0, '미성형'): 4.0, (6.0, '정상'): 70.0, (6.0, '가스'): 3.0, (6.0, '미성형'): 1.0, (7.0, '정상'): 170.0, (7.0, '가스'): 4.0, (7.0, '미성형'): 2.0}


c:\Users\한국전파진흥협회\OneDrive\바탕 화면\taein\스마트팩토리\KAMP 대회\2026년 경진대회\KAMP-IMM\notebooks\rg3_two_type.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d['comparison_block'] = np.nan
c:\Users\한국전파진흥협회\OneDrive\바탕 화면\taein\스마트팩토리\KAMP 대회\2026년 경진대회\KAMP-IMM\notebooks\rg3_two_type.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d['type'] = np.select([r.str.contains('가스'), r.str.contains('미성형')], ['가스', '미성형'], '정상')


In [2]:
P, LOG = run(D)
P.to_csv(f'{OUT}/predictions.csv', index=False, encoding='utf-8-sig')
display(LOG)

완료 leave_one_block_out 3.0
완료 leave_one_block_out 4.0
완료 leave_one_block_out 5.0
완료 leave_one_block_out 6.0
완료 leave_one_block_out 7.0
완료 past_blocks_only 4.0
완료 past_blocks_only 5.0
완료 past_blocks_only 6.0
완료 past_blocks_only 7.0


,scheme,block,train,test,train 가스,train 미성형,test 가스,test 미성형,평가된 모델
0,leave_one_block_out,3.0,550,41,17,8,0,0,"A_gas_std4, B_short_short6, D_all_in_one, C_un..."
1,leave_one_block_out,4.0,488,103,13,7,4,1,"A_gas_std4, B_short_short6, D_all_in_one, C_un..."
2,leave_one_block_out,5.0,394,197,11,4,6,4,"A_gas_std4, B_short_short6, D_all_in_one, C_un..."
3,leave_one_block_out,6.0,517,74,14,7,3,1,"A_gas_std4, B_short_short6, D_all_in_one, C_un..."
4,leave_one_block_out,7.0,415,176,13,6,4,2,"A_gas_std4, B_short_short6, D_all_in_one, C_un..."
5,past_blocks_only,4.0,41,103,0,0,4,1,
6,past_blocks_only,5.0,144,197,4,1,6,4,"A_gas_std4, B_short_short6, D_all_in_one, C_un..."
7,past_blocks_only,6.0,341,74,10,5,3,1,"A_gas_std4, B_short_short6, D_all_in_one, C_un..."
8,past_blocks_only,7.0,415,176,13,6,4,2,"A_gas_std4, B_short_short6, D_all_in_one, C_un..."


## 결과
- **AP 가스 / AP 미성형**: 각 유형 vs 정상만 놓고 계산 (기준선 = 그 유형의 비율)
- **상위10%**: 관측 구간마다 점수 상위 10%를 검사할 때 잡은 수
- **경보**: 학습 정상 99% 임계값을 넘은 샷 (TP 유형별, FP = 정상 오경보)

In [3]:
M = metrics(P); M.to_csv(f'{OUT}/metrics.csv', index=False, encoding='utf-8-sig')
for s in M['scheme'].unique():
    print(s); display(M[M['scheme'] == s].drop(columns='scheme').round(3))

leave_one_block_out


,model,샷,가스,미성형,AP 전체,기준선 전체,AP 가스,기준선 가스,AP 미성형,기준선 미성형,상위10% 가스,상위10% 미성형,상위10% 선택,경보 TP 가스,경보 TP 미성형,경보 FP,정상 중 경보 비율
0,A_gas_std4,591,17,8,0.118,0.042,0.139,0.029,0.015,0.014,6/17,0/8,62,3,0,13,0.023
1,B_short_short6,591,17,8,0.038,0.042,0.027,0.029,0.015,0.014,2/17,0/8,62,0,0,6,0.011
2,D_all_in_one,591,17,8,0.073,0.042,0.069,0.029,0.018,0.014,3/17,0/8,62,1,0,9,0.016
3,C_union_max,591,17,8,0.110,0.042,0.132,0.029,0.013,0.014,5/17,0/8,62,3,0,11,0.019


past_blocks_only


,model,샷,가스,미성형,AP 전체,기준선 전체,AP 가스,기준선 가스,AP 미성형,기준선 미성형,상위10% 가스,상위10% 미성형,상위10% 선택,경보 TP 가스,경보 TP 미성형,경보 FP,정상 중 경보 비율
4,A_gas_std4,447,13,7,0.100,0.045,0.114,0.03,0.016,0.016,4/13,0/7,46,2,0,8,0.019
5,B_short_short6,447,13,7,0.062,0.045,0.051,0.03,0.022,0.016,3/13,1/7,46,1,0,5,0.012
6,D_all_in_one,447,13,7,0.068,0.045,0.067,0.03,0.016,0.016,2/13,0/7,46,1,0,4,0.009
7,C_union_max,447,13,7,0.098,0.045,0.115,0.03,0.015,0.016,5/13,0/7,46,3,0,7,0.016


## 해석 가이드
| 결과 | 판단 |
|---|---|
| A의 가스 AP ≈ 0.139 | 하연님 결과 재현 → 비교 기준 확보 |
| B의 미성형 AP가 기준선보다 뚜렷이 높음 (구간 빼기·과거만 둘 다) | 미성형 변수가 RG3 미성형을 잡음 |
| C가 A만큼 가스를 잡으면서 미성형도 더 잡음 (FP 증가는 작음) | **가스+미성형 통합 모델로 채택 후보** |
| D가 C보다 나음 | 한 모델로 합치는 게 나음 |
| B·C의 미성형 AP가 기준선 수준 | RG3 미성형은 공정값으로 구분 안 됨 → 가스 모델(A) 유지 |
⚠ 미성형이 8샷뿐이라 한두 샷 차이로 결과가 크게 흔들림 → 두 검증 방식 모두에서 같은 방향일 때만 의미 있게 해석